# Customer Churn Prediction — Model Training & Evaluation Pipeline

**Author:** Gokul  
**Algorithms:** Logistic Regression, Random Forest, Gradient Boosting  
**Focus:** Data Preprocessing, Imbalance Handling, Model Comparison, Threshold Optimization & Serialization.

## 1. Import Dependencies & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

# Load & Clean
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).copy()
print(f'Cleaned dataset shape: {df.shape}')

## 2. Feature-Target Separation & Train-Val-Test Split
We use a 60% Train, 20% Validation, 20% Test stratified split to prevent data leakage.

In [ ]:
X = df.drop(columns=['Churn', 'customerID'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

numerical_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numerical_features]

# 60% Train, 20% Val, 20% Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f'Training set:   {X_train.shape}')
print(f'Validation set: {X_val.shape}')
print(f'Test set:       {X_test.shape}')

## 3. Construct Scikit-Learn Preprocessing Pipeline
Features are processed inside a `ColumnTransformer` to ensure test data leakage is strictly avoided.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## 4. Train & Compare Multiple Machine Learning Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}

results = []
trained_pipelines = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    
    y_val_pred = pipe.predict(X_val)
    y_val_prob = pipe.predict_proba(X_val)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_val, y_val_pred),
        'Precision': precision_score(y_val, y_val_pred, zero_division=0),
        'Recall': recall_score(y_val, y_val_pred, zero_division=0),
        'F1 Score': f1_score(y_val, y_val_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_val, y_val_prob)
    })
    trained_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
display(results_df.round(4))

## 5. Model Selection & Final Refit
We select **Logistic Regression** based on top ROC-AUC (0.836) and superior Recall (0.767). We refit on Train + Validation data.

In [ ]:
best_name = results_df.iloc[0]['Model']
best_pipe = trained_pipelines[best_name]
print(f'Selected model: {best_name}')

# Refit on combined Train + Val
best_pipe.fit(X_train_val, y_train_val)

# Evaluate on unseen Test Set
y_test_pred = best_pipe.predict(X_test)
y_test_prob = best_pipe.predict_proba(X_test)[:, 1]

print(f'Test ROC-AUC: {roc_auc_score(y_test, y_test_prob):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_test_pred, target_names=['No Churn', 'Churn']))

## 6. Confusion Matrix & ROC Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, display_labels=['Retained', 'Churned'], cmap='Blues', ax=ax1)
ax1.set_title('Test Confusion Matrix')

RocCurveDisplay.from_predictions(y_test, y_test_prob, ax=ax2, name=best_name)
ax2.plot([0, 1], [0, 1], 'k--', label='Random Chance')
ax2.set_title('Test ROC Curve')
ax2.legend()
plt.tight_layout()
plt.show()

## 7. Model Serialization
Save the complete pipeline into `models/churn_pipeline.joblib`.

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(best_pipe, '../models/churn_pipeline.joblib')
print('Model pipeline saved successfully!')